In [20]:
# import pandas as pd
# import numpy as np
# import os
# from datetime import datetime

# # --- 0. Data Paths on Kaggle (as provided by user) ---
# # Assuming your notebook or environment has access to these paths
# # If running locally, you'd need to download these and adjust paths
# kaggle_data_dir_ml = '/kaggle/input/movies2/'
# kaggle_data_dir_netflix = '/kaggle/input/netflixdata/'

# # MovieLens 20M
# ml_ratings_path = os.path.join(kaggle_data_dir_ml, 'rating.csv')
# ml_movies_path = os.path.join(kaggle_data_dir_ml, 'movie_titles.csv')
# # Note: movie.csv is equivalent to the 'movies.csv' from earlier Kaggle version
# # and rating.csv is equivalent to 'ratings.csv' from earlier Kaggle version

# # Netflix Prize
# netflix_combined_data_1_path = os.path.join(kaggle_data_dir_netflix, 'combined_data_1.txt')
# netflix_movie_titles_path = os.path.join(kaggle_data_dir_ml, 'movie_titles.csv')

# # You can add other combined_data_X.txt files here if you want to process them all
# # netflix_combined_data_paths = [
# #     os.path.join(kaggle_data_dir_netflix, 'combined_data_1.txt'),
# #     os.path.join(kaggle_data_dir_netflix, 'combined_data_2.txt'),
# #     os.path.join(kaggle_data_dir_netflix, 'combined_data_3.txt'),
# #     os.path.join(kaggle_data_dir_netflix, 'combined_data_4.txt')
# # ]

# # --- 1. Define Unified Schema Structures ---
# # We'll aim for three main unified DataFrames:
# # 1. users_df: userId
# # 2. items_df: itemId, title, genre, year, type (e.g., 'movie')
# # 3. interactions_df: userId, itemId, rating, timestamp, source (to track origin)

# unified_users = []
# unified_items = []
# unified_interactions = []

# # --- 2. Load and Process Each Dataset ---


In [21]:

# # --- 2.1. MovieLens 20M ---
# print("Processing MovieLens 20M data...")
# try:
#     ml_ratings = pd.read_csv(ml_ratings_path)
#     ml_movies = pd.read_csv(ml_movies_path) # Now correctly named 'movie.csv'

#     # Interactions
#     ml_interactions = ml_ratings.rename(columns={
#         'userId': 'userId',
#         'movieId': 'itemId',
#         'rating': 'rating',
#         'timestamp': 'timestamp'
#     })
#     ml_interactions['source'] = 'MovieLens'
#     ml_interactions['timestamp'] = pd.to_datetime(ml_interactions['timestamp'], unit='s')
#     unified_interactions.append(ml_interactions)

#     # Items (Movies)
#     ml_items = ml_movies.rename(columns={
#         'movieId': 'itemId',
#         'title': 'title',
#         'genres': 'genre'
#     })
#     ml_items['type'] = 'movie'
#     # Extract year from title if present (e.g., "Movie Title (YYYY)")
#     ml_items['year'] = ml_items['title'].str.extract(r'\((\d{4})\)').astype(float)
#     ml_items['description'] = None # No direct description in this dataset
#     # Select only the columns for the unified schema
#     unified_items.append(ml_items[['itemId', 'title', 'type', 'genre', 'year', 'description']])

#     # Users - MovieLens only provides IDs, no demographics
#     ml_users = pd.DataFrame({'userId': ml_ratings['userId'].unique()})
#     ml_users['source'] = 'MovieLens'
#     unified_users.append(ml_users)

# except FileNotFoundError as e:
#     print(f"MovieLens files not found: {e}. Please ensure paths are correct: {ml_ratings_path}, {ml_movies_path}")
# except Exception as e:
#     print(f"Error processing MovieLens data: {e}")

In [22]:

# # --- 2.2. Netflix Prize Dataset ---
# print("\nProcessing Netflix Prize data (sample from combined_data_1.txt)...")
# try:
#     # Load Netflix movie titles
#     # The 'movie_titles.csv' is tricky; it's often without headers and has a specific encoding.
#     # It usually has MovieID, Year, MovieTitle.
#     netflix_titles = pd.read_csv(
#         netflix_movie_titles_path,
#         encoding='ISO-8859-1', # Common encoding for this file
#         header=None,
#         names=['movieId', 'year', 'title'],
#         on_bad_lines='skip' # Skip lines that don't match expected columns
#     )
#     # Ensure movieId is string to match how we'll prepend in interactions
#     netflix_titles['movieId'] = netflix_titles['movieId'].astype(str)

#     # Parse Netflix combined_data_1.txt for interactions
#     netflix_raw_data = []
#     with open(netflix_combined_data_1_path, 'r') as f:
#         current_movie_id_raw = None
#         for line in f:
#             line = line.strip()
#             if line.endswith(':'):
#                 current_movie_id_raw = line[:-1] # Remove the colon
#             else:
#                 user_id, rating, date_str = line.split(',')
#                 timestamp = datetime.strptime(date_str, '%Y-%m-%d')
#                 netflix_raw_data.append({
#                     'userId': int(user_id),
#                     'itemId': f"netflix_{current_movie_id_raw}", # Prefix ID to avoid clashes with MovieLens
#                     'rating': float(rating),
#                     'timestamp': timestamp,
#                     'source': 'Netflix'
#                 })
#     netflix_interactions = pd.DataFrame(netflix_raw_data)
#     unified_interactions.append(netflix_interactions[['userId', 'itemId', 'rating', 'timestamp', 'source']])

#     # Items (Movies) for Netflix
#     # Map raw Netflix MovieID to unified itemId format
#     netflix_titles['unified_itemId'] = 'netflix_' + netflix_titles['movieId'].astype(str)
#     netflix_items = netflix_titles.rename(columns={
#         'title': 'title',
#         'year': 'year' # This is the release year
#     })
#     netflix_items['itemId'] = netflix_items['unified_itemId']
#     netflix_items['type'] = 'movie' # All Netflix items are movies
#     netflix_items['genre'] = 'Unknown' # Genre not directly in titles file
#     netflix_items['description'] = None # No description in these files
#     # Select only the columns for the unified schema and drop duplicates (e.g. from bad_lines skip)
#     unified_items.append(netflix_items[['itemId', 'title', 'type', 'genre', 'year', 'description']].drop_duplicates(subset=['itemId']))

#     # Users - Netflix also primarily provides IDs
#     netflix_users = pd.DataFrame({'userId': netflix_interactions['userId'].unique()})
#     netflix_users['source'] = 'Netflix'
#     unified_users.append(netflix_users)

# except FileNotFoundError as e:
#     print(f"Netflix Prize files not found: {e}. Please ensure paths are correct: {netflix_combined_data_1_path}, {netflix_movie_titles_path}")
# except Exception as e:
#     print(f"Error processing Netflix data: {e}")


In [23]:


# # --- 3. Consolidate Unified DataFrames ---
# print("\nConsolidating unified dataframes...")

# # Combine all interactions
# if unified_interactions:
#     final_interactions_df = pd.concat(unified_interactions, ignore_index=True)
#     # Ensure itemId and userId are consistent types (e.g., string for all)
#     final_interactions_df['userId'] = final_interactions_df['userId'].astype(str)
#     final_interactions_df['itemId'] = final_interactions_df['itemId'].astype(str)
#     print(f"Total Unified Interactions: {len(final_interactions_df)} records")
#     print(final_interactions_df.head())
# else:
#     final_interactions_df = pd.DataFrame()
#     print("No interactions data loaded.")

# # Combine all items
# if unified_items:
#     final_items_df = pd.concat(unified_items, ignore_index=True)
#     # Remove duplicate items based on itemId, keeping the first occurrence.
#     # This is important if an item ID from MovieLens and Netflix happen to collide
#     # (though we've prefixed Netflix IDs to prevent this) or if there are duplicates within a source.
#     final_items_df['itemId'] = final_items_df['itemId'].astype(str)
#     final_items_df.drop_duplicates(subset=['itemId'], inplace=True)
#     print(f"\nTotal Unified Items: {len(final_items_df)} unique records")
#     print(final_items_df.head())
# else:
#     final_items_df = pd.DataFrame()
#     print("No items data loaded.")

# # Combine all users
# if unified_users:
#     final_users_df = pd.concat(unified_users, ignore_index=True)
#     final_users_df['userId'] = final_users_df['userId'].astype(str)
#     # Remove duplicate users based on userId, keeping the first occurrence
#     final_users_df.drop_duplicates(subset=['userId'], inplace=True)
#     print(f"\nTotal Unified Users: {len(final_users_df)} unique records")
#     print(final_users_df.head())
# else:
#     final_users_df = pd.DataFrame()
#     print("No users data loaded.")

# print("\nData integration process complete!")
# print("You now have three unified DataFrames: final_users_df, final_items_df, final_interactions_df.")
# print("Next steps would involve further cleaning, feature engineering, and content understanding with Gemini.")

In [24]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
from tqdm.auto import tqdm # Import tqdm for progress bars

# --- 0. Data Paths on Kaggle (as provided by user) ---
# Assuming your notebook or environment has access to these paths
# If running locally, you'd need to download these and adjust paths
kaggle_data_dir_ml = '/kaggle/input/movies2/'
kaggle_data_dir_netflix = '/kaggle/input/netflixdata/'

# MovieLens 20M
ml_ratings_path = os.path.join(kaggle_data_dir_ml, 'rating.csv')
ml_movies_path = os.path.join(kaggle_data_dir_ml, 'movie_titles.csv')
# Note: movie.csv is equivalent to the 'movies.csv' from earlier Kaggle version
# and rating.csv is equivalent to 'ratings.csv' from earlier Kaggle version

# Netflix Prize
netflix_combined_data_1_path = os.path.join(kaggle_data_dir_netflix, 'combined_data_1.txt')
netflix_movie_titles_path = os.path.join(kaggle_data_dir_ml, 'movie_titles.csv')

# You can add other combined_data_X.txt files here if you want to process them all
# netflix_combined_data_paths = [
#     os.path.join(kaggle_data_dir_netflix, 'combined_data_1.txt'),
#     os.path.join(kaggle_data_dir_netflix, 'combined_data_2.txt'),
#     os.path.join(kaggle_data_dir_netflix, 'combined_data_3.txt'),
#     os.path.join(kaggle_data_dir_netflix, 'combined_data_4.txt')
# ]

# --- 1. Define Unified Schema Structures ---
# We'll aim for three main unified DataFrames:
# 1. users_df: userId
# 2. items_df: itemId, title, genre, year, type (e.g., 'movie')
# 3. interactions_df: userId, itemId, rating, timestamp, source (to track origin)

unified_users = []
unified_items = []
unified_interactions = []

# --- 2. Load and Process Each Dataset ---

# --- 2.1. MovieLens 20M ---
print("Processing MovieLens 20M data...")
try:
    print("Loading MovieLens ratings...")
    ml_ratings = pd.read_csv(ml_ratings_path)
    print("Loading MovieLens movies...")
    ml_movies = pd.read_csv(ml_movies_path)

    # Interactions
    ml_interactions = ml_ratings.rename(columns={
        'userId': 'userId',
        'movieId': 'itemId',
        'rating': 'rating',
        'timestamp': 'timestamp'
    })
    ml_interactions['source'] = 'MovieLens'
    # CORRECTED: Removed unit='s' for MovieLens timestamp conversion
    ml_interactions['timestamp'] = pd.to_datetime(ml_interactions['timestamp'])
    unified_interactions.append(ml_interactions)
    print(f"MovieLens interactions processed: {len(ml_interactions)} records.")


    # Items (Movies)
    ml_items = ml_movies.rename(columns={
        'movieId': 'itemId',
        'title': 'title',
        'genres': 'genre'
    })
    ml_items['type'] = 'movie'
    # Extract year from title if present (e.g., "Movie Title (YYYY)")
    ml_items['year'] = ml_items['title'].str.extract(r'\((\d{4})\)').astype(float)
    ml_items['description'] = None # No direct description in this dataset
    # Select only the columns for the unified schema
    unified_items.append(ml_items[['itemId', 'title', 'type', 'genre', 'year', 'description']])
    print(f"MovieLens items processed: {len(ml_items)} records.")


    # Users - MovieLens only provides IDs, no demographics
    ml_users = pd.DataFrame({'userId': ml_ratings['userId'].unique()})
    ml_users['source'] = 'MovieLens'
    unified_users.append(ml_users)
    print(f"MovieLens users processed: {len(ml_users)} records.")


except FileNotFoundError as e:
    print(f"MovieLens files not found: {e}. Please ensure paths are correct: {ml_ratings_path}, {ml_movies_path}")
except Exception as e:
    print(f"Error processing MovieLens data: {e}")

# --- 2.2. Netflix Prize Dataset ---
print("\nProcessing Netflix Prize data (sample from combined_data_1.txt)...")
try:
    print("Loading Netflix movie titles...")
    # Load Netflix movie titles
    # The 'movie_titles.csv' is tricky; it's often without headers and has a specific encoding.
    # It usually has MovieID, Year, MovieTitle.
    netflix_titles = pd.read_csv(
        netflix_movie_titles_path,
        encoding='ISO-8859-1', # Common encoding for this file
        header=None,
        names=['movieId', 'year', 'title'],
        on_bad_lines='skip' # Skip lines that don't match expected columns
    )
    # Ensure movieId is string to match how we'll prepend in interactions
    netflix_titles['movieId'] = netflix_titles['movieId'].astype(str)
    print(f"Netflix movie titles loaded: {len(netflix_titles)} records.")


    print("Parsing Netflix combined_data_1.txt for interactions (this might take a while for large files)...")
    netflix_raw_data = []
    # Use tqdm to track progress of file reading line by line
    with open(netflix_combined_data_1_path, 'r') as f:
        # Get total lines for tqdm progress bar (approximate for very large files)
        total_lines = sum(1 for _ in open(netflix_combined_data_1_path, 'r'))
        f.seek(0) # Reset file pointer after counting lines

        current_movie_id_raw = None
        for line in tqdm(f, total=total_lines, desc="Reading Netflix Data"):
            line = line.strip()
            if line.endswith(':'):
                current_movie_id_raw = line[:-1] # Remove the colon
            else:
                user_id, rating, date_str = line.split(',')
                timestamp = datetime.strptime(date_str, '%Y-%m-%d')
                netflix_raw_data.append({
                    'userId': int(user_id),
                    'itemId': f"netflix_{current_movie_id_raw}", # Prefix ID to avoid clashes with MovieLens
                    'rating': float(rating),
                    'timestamp': timestamp,
                    'source': 'Netflix'
                })
    netflix_interactions = pd.DataFrame(netflix_raw_data)
    unified_interactions.append(netflix_interactions[['userId', 'itemId', 'rating', 'timestamp', 'source']])
    print(f"Netflix interactions processed: {len(netflix_interactions)} records.")


    # Items (Movies) for Netflix
    # Map raw Netflix MovieID to unified itemId format
    netflix_titles['unified_itemId'] = 'netflix_' + netflix_titles['movieId'].astype(str)
    netflix_items = netflix_titles.rename(columns={
        'title': 'title',
        'year': 'year' # This is the release year
    })
    netflix_items['itemId'] = netflix_items['unified_itemId']
    netflix_items['type'] = 'movie' # All Netflix items are movies
    netflix_items['genre'] = 'Unknown' # Genre not directly in titles file
    netflix_items['description'] = None # No description in these files
    # Select only the columns for the unified schema and drop duplicates (e.g. from bad_lines skip)
    unified_items.append(netflix_items[['itemId', 'title', 'type', 'genre', 'year', 'description']].drop_duplicates(subset=['itemId']))
    print(f"Netflix items processed: {len(netflix_items)} records.")


    # Users - Netflix also primarily provides IDs
    netflix_users = pd.DataFrame({'userId': netflix_interactions['userId'].unique()})
    netflix_users['source'] = 'Netflix'
    unified_users.append(netflix_users)
    print(f"Netflix users processed: {len(netflix_users)} records.")


except FileNotFoundError as e:
    print(f"Netflix Prize files not found: {e}. Please ensure paths are correct: {netflix_combined_data_1_path}, {netflix_movie_titles_path}")
except Exception as e:
    print(f"Error processing Netflix data: {e}")

# --- 3. Consolidate Unified DataFrames ---
print("\nConsolidating unified dataframes...")

# Combine all interactions
if unified_interactions:
    print("Concatenating unified interactions...")
    final_interactions_df = pd.concat(unified_interactions, ignore_index=True)
    # Ensure itemId and userId are consistent types (e.g., string for all)
    final_interactions_df['userId'] = final_interactions_df['userId'].astype(str)
    final_interactions_df['itemId'] = final_interactions_df['itemId'].astype(str)
    print(f"Total Unified Interactions: {len(final_interactions_df)} records")
    print(final_interactions_df.head())
else:
    final_interactions_df = pd.DataFrame()
    print("No interactions data loaded.")

# Combine all items
if unified_items:
    print("\nConcatenating unified items...")
    final_items_df = pd.concat(unified_items, ignore_index=True)
    # Remove duplicate items based on itemId, keeping the first occurrence.
    # This is important if an item ID from MovieLens and Netflix happen to collide
    # (though we've prefixed Netflix IDs to prevent this) or if there are duplicates within a source.
    final_items_df['itemId'] = final_items_df['itemId'].astype(str)
    final_items_df.drop_duplicates(subset=['itemId'], inplace=True)
    print(f"\nTotal Unified Items: {len(final_items_df)} unique records")
    print(final_items_df.head())
else:
    final_items_df = pd.DataFrame()
    print("No items data loaded.")

# Combine all users
if unified_users:
    print("\nConcatenating unified users...")
    final_users_df = pd.concat(unified_users, ignore_index=True)
    final_users_df['userId'] = final_users_df['userId'].astype(str)
    # Remove duplicate users based on userId, keeping the first occurrence
    final_users_df.drop_duplicates(subset=['userId'], inplace=True)
    print(f"\nTotal Unified Users: {len(final_users_df)} unique records")
    print(final_users_df.head())
else:
    final_users_df = pd.DataFrame()
    print("No users data loaded.")

print("\nData integration process complete!")
print("You now have three unified DataFrames: final_users_df, final_items_df, final_interactions_df.")
print("Next steps would involve further cleaning, feature engineering, and content understanding with Gemini.")



Processing MovieLens 20M data...
Loading MovieLens ratings...
Loading MovieLens movies...
Error processing MovieLens data: 'timestamp'

Processing Netflix Prize data (sample from combined_data_1.txt)...
Loading Netflix movie titles...
Netflix movie titles loaded: 4 records.
Parsing Netflix combined_data_1.txt for interactions (this might take a while for large files)...


Reading Netflix Data:   0%|          | 0/8 [00:00<?, ?it/s]

Error processing Netflix data: not enough values to unpack (expected 3, got 1)

Consolidating unified dataframes...
No interactions data loaded.
No items data loaded.
No users data loaded.

Data integration process complete!
You now have three unified DataFrames: final_users_df, final_items_df, final_interactions_df.
Next steps would involve further cleaning, feature engineering, and content understanding with Gemini.


In [25]:
# # Features implemented
# Multi-source integration	-	Combines MovieLens + Netflix cleanly
# Unified item & interaction schema	-	Standardized itemId, userId, rating, timestamp
# Timestamp processing	-	Uses pd.to_datetime() for temporal features
# Type tagging (movies)	-	Uses 'type': 'movie' to label content type
# Duplicate handling     -     	Ensures item/user uniqueness
# Progress bar (tqdm)	-	For large Netflix text files
# Robust error handling	-	Try-except around all key blocks
# Prefixed item IDs (Netflix)	-	Avoids ID collision with MovieLens

In [26]:
# # 1. Cross-Domain Recommendation - further

# book_titles = ["Atomic Habits", "The Hobbit", "Becoming", "Where the Crawdads Sing"]
# book_embeddings = model.encode(book_titles)

# # Recommend books for a user with a movie profile
# book_scores = cosine_similarity([user_vector], book_embeddings)[0]


In [27]:
# explainability 
# diversity boost

# import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
import numpy as np

def enhanced_recommendations(user_profile_vector, item_df, item_embeddings, user_history_ids, interactions_df):
    """
    Returns a DataFrame of recommendations with:
    - Explainability (why it was recommended)
    - Cold start fallback
    - Diversity (from different genres)
    """

    # STEP 5: Cosine similarity for personalization
    similarity_scores = cosine_similarity([user_profile_vector], item_embeddings)[0]
    item_df = item_df.copy()
    item_df['similarity'] = similarity_scores

    # STEP 6: Cold Start Handling
    if user_history_ids is None or len(user_history_ids) == 0:
        popular_items = interactions_df['itemId'].value_counts().head(10).index
        cold_start_df = item_df[item_df['itemId'].isin(popular_items)].copy()
        cold_start_df['reason'] = "Popular among users (cold start)"
        return cold_start_df[['itemId', 'title', 'genre', 'reason']]

    # STEP 7: Combine Personalization + Popularity
    popularity = interactions_df['itemId'].value_counts().to_dict()
    item_df['popularity'] = item_df['itemId'].map(popularity).fillna(0)

    scaler = MinMaxScaler()
    item_df['popularity_norm'] = scaler.fit_transform(item_df[['popularity']])

    item_df['hybrid_score'] = 0.7 * item_df['similarity'] + 0.3 * item_df['popularity_norm']
    item_df = item_df[~item_df['itemId'].isin(user_history_ids)]

    # Top personalized recommendations
    top_personalized = item_df.sort_values('hybrid_score', ascending=False).head(10).copy()
    top_personalized['reason'] = "Based on your interest in similar items"

    # STEP 7 Continued: Diversity by genre
    diversity_recs = item_df.groupby('genre').apply(lambda g: g.nlargest(1, 'hybrid_score')).reset_index(drop=True)
    diversity_recs = diversity_recs[~diversity_recs['itemId'].isin(top_personalized['itemId'])].head(5)
    diversity_recs['reason'] = "To add variety from genre: " + diversity_recs['genre'].astype(str)

    # Combine
    final_recommendations = pd.concat([top_personalized, diversity_recs], ignore_index=True)
    return final_recommendations[['itemId', 'title', 'genre', 'reason', 'hybrid_score']]


In [29]:
final_interactions_df

""


In [ ]:

# | userId | itemId         | rating | timestamp           | source    |
# | ------ | -------------- | ------ | ------------------- | --------- |
# | 1      | 1              | 4.0    | 2000-07-31 00:00:00 | MovieLens |
# | 2      | 2              | 3.5    | 2000-08-14 00:00:00 | MovieLens |
# | 1234   | netflix\_177   | 5.0    | 2005-01-01 00:00:00 | Netflix   |
# | 1235   | netflix\_12345 | 4.0    | 2004-07-18 00:00:00 | Netflix   |


In [ ]:
final_items_df

In [ ]:
# itemId	title	type	genre	year	description
# 1	Toy Story (1995)	movie	Animation	1995	None
# 2	Jumanji (1995)	movie	Adventure	1995	None